# UnifiedTMIL v3 — End-to-End Pipeline
## Single Forward: Account Detection + Transaction Localization
### Architecture: BERT4ETH → TCN → Gated Attention → MLP Classifier
### Features: LOO Reputation Only (no position, no engineered features)


In [ ]:
# ⚙️ CONFIG — EDIT THESE
QUICK = False  # True=2 seeds, 4 epochs; False=5 seeds, 10 epochs
EPOCHS = 4 if QUICK else 10
SEEDS = [42, 1337] if QUICK else [42, 1337, 7, 99, 2024]
BS = 64
LR = 1e-3
EMBED_DIM = 64
EXTRA_FEAT_DIM = 1  # 0=none, 1=rep_only, 2=rep+freq


In [ ]:
import sys, os, json, pickle, random, time, copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score, average_precision_score


In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")

## Data Loading

In [ ]:
# ── Mount Google Drive or use Kaggle dataset ──
# If running on Kaggle, data will be at /kaggle/input/
# If running on Colab, mount drive and point to data
import pathlib
POSSIBLE_PATHS = [
    ".",
    "/kaggle/input/successkaboom",
    "/kaggle/input/successkaboom-data",
    "/content/drive/MyDrive/Successkaboom",
]
ROOT = None
for p in POSSIBLE_PATHS:
    if os.path.exists(os.path.join(p, "data/bert4eth/vocab.pkl")):
        ROOT = p
        break
if ROOT is None:
    ROOT = os.getcwd()  # fallback
print(f"Root: {ROOT}")

In [ ]:
class Vocab:
    """Bidirectional mapping between address tokens and integer IDs."""
    def __init__(self):
        self.token2id = {"[PAD]": 0, "[UNK]": 1}
        self.id2token = {0: "[PAD]", 1: "[UNK]"}
    def add(self, token):
        if token not in self.token2id:
            idx = len(self.token2id)
            self.token2id[token] = idx
            self.id2token[idx] = token
    def get_id(self, token):
        return self.token2id.get(token, 1)
    def __len__(self):
        return len(self.token2id)

sys.modules["__main__"].Vocab = Vocab

def load_pkl(path):
    return pickle.load(open(path, "rb"))

SRC = os.path.join(ROOT, "data/bert4eth")
DATA = os.path.join(ROOT, "data")
vocab = load_pkl(os.path.join(SRC, "vocab.pkl"))
train_bags = load_pkl(os.path.join(SRC, "train_bags.pkl"))
test_bags = load_pkl(os.path.join(SRC, "test_bags.pkl"))
ptx_bags = load_pkl(os.path.join(DATA, "ptx_bags.pkl"))
defi_bags = load_pkl(os.path.join(DATA, "defi_hard_bags.pkl"))
normal_neg = load_pkl(os.path.join(DATA, "normal_eoa_neg.pkl"))
vocab_size = len(vocab.token2id)
print(f"vocab={vocab_size}, train={len(train_bags)}, test={len(test_bags)}, ptx={len(ptx_bags)}, defi={len(defi_bags)}, normal_neg={len(normal_neg)}")

## Index Map & LOO Reputation (Leakage-Free)

In [ ]:
all_bags = train_bags + test_bags + ptx_bags + defi_bags + normal_neg
n_tr = len(train_bags); n_te = len(test_bags)
tr_idx = list(range(n_tr))
te_idx = list(range(n_tr, n_tr + n_te))
ptx_start = n_tr + n_te
ptx_idx = list(range(ptx_start, ptx_start + len(ptx_bags)))
defi_start = ptx_start + len(ptx_bags)
defi_idx = list(range(defi_start, defi_start + len(defi_bags)))
norm_start = defi_start + len(defi_bags)
norm_idx = list(range(norm_start, norm_start + len(normal_neg)))

# LOO Reputation
cp_phish = {}; cp_total = {}; bag_cps = []
for bag in all_bags:
    cps = set(bag["input_ids"][:bag["length"]]); cps.discard(0)
    bag_cps.append(cps)
    for cp in cps:
        cp_total[cp] = cp_total.get(cp, 0) + 1
        if bag["label"] == 1: cp_phish[cp] = cp_phish.get(cp, 0) + 1

loo_rep = []
for i, (bag, cps) in enumerate(zip(all_bags, bag_cps)):
    rep_i = {}
    for cp in cps:
        total = cp_total.get(cp, 0)
        phish = cp_phish.get(cp, 0)
        phish_loo = phish - (1 if bag["label"] == 1 else 0)
        total_loo = total - 1
        rep_i[cp] = phish_loo / total_loo if total_loo > 0 else 0.0
    loo_rep.append(rep_i)

# Cross-domain & Hard
ptx_pos = [(b, i) for b, i in zip(ptx_bags, ptx_idx) if b["label"] == 1]
xd_bags = [b for b, _ in ptx_pos] + normal_neg
xd_idx = [i for _, i in ptx_pos] + norm_idx
hard_be = [b for b, _ in ptx_pos] + defi_bags
hard_ie = [i for _, i in ptx_pos] + defi_idx
print(f"Cross-domain: {len(xd_bags)} bags, Hard: {len(hard_be)} bags")

# Loc bags (interior GT only)
loc_bags = []; loc_idx = []
for bag, bidx in zip(ptx_bags, ptx_idx):
    if bag["label"] == 1 and bag.get("gt_idx"):
        L = bag["length"]
        if any(g < L - 1 for g in bag["gt_idx"]):
            loc_bags.append(bag); loc_idx.append(bidx)
print(f"Loc bags: {len(loc_bags)} (interior GT only)")

## V3 Model Architecture (Pure Neural, End-to-End)

In [ ]:
class UnifiedTMILv3(nn.Module):
    """v3: Pure neural, no GBM/LGBM, no engineered features, no position.
    
    Changes from v1:
        - Removed GBM/LambdaMART (end-to-end neural)
        - Removed all positional features (prel, dist_nl, rank, z-score)
        - Removed engineered features (spike, novelty, cummax, runmax, etc.)
        - Extra_feat_dim=1: only LOO reputation injected at attention
        - Learnable temperature for attention sharpness
        - Contrastive loss + Entropy regularization for localization
    """
    def __init__(self, vocab_size, embed_dim=64, extra_feat_dim=1):
        super().__init__()
        self.cp_embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.io_embed = nn.Embedding(3, embed_dim, padding_idx=0)
        self.hc_proj = nn.Sequential(
            nn.Linear(2, embed_dim), nn.LayerNorm(embed_dim), nn.ReLU()
        )
        self.tcn = nn.Conv1d(embed_dim, embed_dim, 3, padding=1)
        self.norm = nn.LayerNorm(embed_dim)
        attn_in = embed_dim + extra_feat_dim
        self.attn_V = nn.Linear(attn_in, 128)
        self.attn_U = nn.Linear(attn_in, 128)
        self.attn_w = nn.Linear(128, 1)
        self.log_tau = nn.Parameter(torch.zeros(1))
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 1)
        )
    
    def forward(self, ids, io, hc, mask, extra=None):
        h = self.cp_embed(ids) + self.io_embed(io) + self.hc_proj(hc)
        h = self.norm(h)
        h = h + self.tcn(h.transpose(1, 2)).transpose(1, 2)
        attn_in = torch.cat([h, extra], dim=-1) if extra is not None else h
        tau = torch.exp(self.log_tau).clamp(0.1, 10.0)
        s = self.attn_w(
            torch.tanh(self.attn_V(attn_in)) * torch.sigmoid(self.attn_U(attn_in))
        ).squeeze(-1)
        s = (s / tau).masked_fill(~mask, -1e9)
        a = F.softmax(s, dim=1)
        z = torch.bmm(a.unsqueeze(1), h).squeeze(1)
        return self.classifier(z).squeeze(-1), a

## Loss Functions

In [ ]:
def focal_loss(logits, targets, gamma=2.0):
    bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    return (((1 - torch.exp(-bce)) ** gamma) * bce).mean()

def contrastive_loss(z_pos, z_neg, margin=1.0):
    """Push positive and negative context vectors apart."""
    if len(z_pos) == 0 or len(z_neg) == 0:
        return torch.tensor(0.0, device=z_pos.device if len(z_pos) > 0 else z_neg.device)
    d = F.pairwise_distance(z_pos.mean(0, keepdim=True), z_neg.mean(0, keepdim=True))
    return F.relu(margin - d).mean()

def entropy_reg(attention_weights, mask):
    """Encourage peaked attention for better localization."""
    eps = 1e-8
    return -(attention_weights * torch.log(attention_weights + eps)).sum(1).mean()

## Training Utilities

In [ ]:
def collate(bags, bag_indices, loo_rep, cp_total, extra_feat_dim=1):
    """Collate bags into tensors. V3: extra_feat_dim=1 (LOO rep only)."""
    L = max(b["length"] for b in bags)
    B = len(bags)
    ids = torch.zeros(B, L, dtype=torch.long, device=DEVICE)
    io = torch.zeros(B, L, dtype=torch.long, device=DEVICE)
    hc = torch.zeros(B, L, 2, device=DEVICE)
    mask = torch.zeros(B, L, dtype=torch.bool, device=DEVICE)
    y = torch.zeros(B, device=DEVICE)
    extra = torch.zeros(B, L, extra_feat_dim, device=DEVICE) if extra_feat_dim > 0 else None
    for i, (bag, bidx) in enumerate(zip(bags, bag_indices)):
        n = min(bag["length"], L)
        ids[i, :n] = torch.tensor(bag["input_ids"][:n], dtype=torch.long)
        io[i, :n] = torch.tensor(bag["input_io"][:n], dtype=torch.long)
        amt = np.array(bag["input_amounts"][:n], dtype=np.float32)
        dt = np.array(bag["delta_ts"][:n], dtype=np.float32)
        hc[i, :n, 0] = torch.tensor(np.log1p(np.abs(amt)) * np.sign(amt))
        hc[i, :n, 1] = torch.tensor(np.log1p(dt))
        mask[i, :n] = True
        y[i] = bag["label"]
        if extra is not None and bidx < len(loo_rep):
            rep_dict = loo_rep[bidx]
            for k, cp in enumerate(bag["input_ids"][:n]):
                extra[i, k, 0] = rep_dict.get(cp, 0.0)
                if extra_feat_dim >= 2:
                    extra[i, k, 1] = np.log1p(cp_total.get(cp, 0))
    return ids, io, hc, mask, y, extra

def iterate(bags, bag_indices, bs, rng):
    idx = list(range(len(bags)))
    rng.shuffle(idx)
    for start in range(0, len(idx), bs):
        chunk = idx[start:start + bs]
        yield [bags[i] for i in chunk], [bag_indices[i] for i in chunk]

In [ ]:
def train_one(model, train_bags, tr_idx, loo_rep, cp_total,
              epochs, bs, lr, seed, lc, le, use_focal, gamma, extra_feat_dim):
    rng = random.Random(seed)
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    model.train()
    for ep in range(epochs):
        ep_losses = []
        for batch, bidx in iterate(train_bags, tr_idx, bs, rng):
            ids, io, hc, mask, y, extra = collate(batch, bidx, loo_rep, cp_total, extra_feat_dim)
            logit, a = model(ids, io, hc, mask, extra)
            loss = (focal_loss(logit, y, gamma) if use_focal 
                    else F.binary_cross_entropy_with_logits(logit, y))
            if le > 0:
                loss = loss + le * entropy_reg(a, mask)
            if lc > 0:
                pm, nm = (y == 1), (y == 0)
                if pm.any() and nm.any():
                    with torch.no_grad():
                        h0 = model.cp_embed(ids) + model.io_embed(io) + model.hc_proj(hc)
                        h0 = model.norm(h0)
                    z = torch.bmm(a.unsqueeze(1), h0).squeeze(1)
                    loss = loss + lc * contrastive_loss(z[pm], z[nm])
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            ep_losses.append(loss.item())
        scheduler.step()
        if ep % 2 == 0:
            print(f"    Epoch {ep}: loss={np.mean(ep_losses):.4f}", flush=True)
    return model

## Evaluation Functions

In [ ]:
def eval_account(model, bags, indices, loo_rep, cp_total, extra_feat_dim=1):
    model.eval()
    with torch.no_grad():
        ids, io, hc, mask, y, extra = collate(bags, indices, loo_rep, cp_total, extra_feat_dim)
        logit, _ = model(ids, io, hc, mask, extra)
        prob = torch.sigmoid(logit).cpu().numpy()
        y_np = y.cpu().numpy()
    p, r, f1, _ = precision_recall_fscore_support(
        y_np, (prob >= 0.5).astype(int), average="binary", zero_division=0
    )
    auc = roc_auc_score(y_np, prob) if len(np.unique(y_np)) > 1 else 0.5
    aupr = average_precision_score(y_np, prob) if y_np.sum() > 0 else 0.0
    return {"f1": float(f1), "prec": float(p), "rec": float(r),
            "auc": float(auc), "aupr": float(aupr)}

def eval_localization(model, loc_bags, loc_idx, loo_rep, cp_total, extra_feat_dim=1):
    """V3: Attention-as-ranking (no GBM)."""
    model.eval()
    hits1, hits5, hits10, mrrs = [], [], [], []
    with torch.no_grad():
        for bag, bidx in zip(loc_bags, loc_idx):
            L = bag["length"]
            if L < 2: continue
            gt_cands = [g for g in bag.get("gt_idx", []) if g < L - 1]
            if not gt_cands: continue
            ids, io, hc, mask, _, extra = collate([bag], [bidx], loo_rep, cp_total, extra_feat_dim)
            _, a = model(ids, io, hc, mask, extra)
            scores = a[0, :L].cpu().numpy()
            candidates = list(range(L - 1))
            ranked = sorted(candidates, key=lambda c: -scores[c])
            best_rank = min((ranked.index(g) + 1 for g in gt_cands if g in ranked),
                          default=len(ranked) + 1)
            hits1.append(1 if best_rank <= 1 else 0)
            hits5.append(1 if best_rank <= 5 else 0)
            hits10.append(1 if best_rank <= 10 else 0)
            mrrs.append(1.0 / best_rank)
    if not hits1:
        return {"hit@1": 0.0, "hit@5": 0.0, "hit@10": 0.0, "mrr": 0.0, "n": 0}
    return {
        "hit@1": float(np.mean(hits1)), "hit@5": float(np.mean(hits5)),
        "hit@10": float(np.mean(hits10)), "mrr": float(np.mean(mrrs)), "n": len(hits1)
    }

def ci95(vals):
    vals = np.array(vals)
    rng = np.random.RandomState(42)
    boots = [np.mean(rng.choice(vals, len(vals))) for _ in range(5000)]
    return float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))

def stats(vals):
    vals = np.array(vals)
    lo, hi = ci95(vals)
    return {"mean": float(np.mean(vals)), "std": float(np.std(vals)),
            "ci_low": lo, "ci_high": hi}

## Full Experiment Runner

In [ ]:
def run_exp(lc, le, use_focal=False, gamma=2.0, extra_feat_dim=1, tag=""):
    """Run N-seed experiment with given hyperparameters."""
    id_f1s, id_aucs, x_aucs, hard_aucs = [], [], [], []
    hit1s, hit5s, hit10s, mrrs = [], [], [], []
    
    for seed in SEEDS:
        model = UnifiedTMILv3(vocab_size, embed_dim=EMBED_DIM, extra_feat_dim=extra_feat_dim).to(DEVICE)
        model = train_one(model, train_bags, tr_idx, loo_rep, cp_total,
                         EPOCHS, BS, LR, seed, lc, le, use_focal, gamma, extra_feat_dim)
        r_id = eval_account(model, test_bags, te_idx, loo_rep, cp_total, extra_feat_dim)
        id_f1s.append(r_id["f1"]); id_aucs.append(r_id["auc"])
        r_x = eval_account(model, xd_bags, xd_idx, loo_rep, cp_total, extra_feat_dim)
        x_aucs.append(r_x["auc"])
        r_h = eval_account(model, hard_be, hard_ie, loo_rep, cp_total, extra_feat_dim)
        hard_aucs.append(r_h["auc"])
        loc = eval_localization(model, loc_bags, loc_idx, loo_rep, cp_total, extra_feat_dim)
        hit1s.append(loc["hit@1"]); hit5s.append(loc["hit@5"])
        hit10s.append(loc["hit@10"]); mrrs.append(loc["mrr"])
        print(f"    seed={seed}: F1={r_id["f1"]:.4f} Hard-AUC={r_h["auc"]:.4f} Hit@1={loc["hit@1"]:.4f}")
    
    res = {
        "id_f1": stats(id_f1s), "id_auc": stats(id_aucs),
        "x_auc": stats(x_aucs), "hard_auc": stats(hard_aucs),
        "hit@1": stats(hit1s), "hit@5": stats(hit5s),
        "hit@10": stats(hit10s), "mrr": stats(mrrs),
        "n_seeds": len(SEEDS), "tag": tag,
        "lc": lc, "le": le, "use_focal": use_focal, "gamma": gamma,
        "extra_feat_dim": extra_feat_dim
    }
    return res

## Phase 1: Baseline + λc, λe Sweep

In [ ]:
results = {}
t0 = time.time()
print("=" * 70)
print("PHASE 1: Baseline (no contrastive, no entropy)")
print("=" * 70)
results["baseline"] = run_exp(lc=0.0, le=0.0, tag="baseline")
r = results["baseline"]
print(f"  F1={r["id_f1"]["mean"]:.4f}±{r["id_f1"]["std"]:.4f} "
      f"X-AUC={r["x_auc"]["mean"]:.4f} Hard={r["hard_auc"]["mean"]:.4f} "
      f"Hit@1={r["hit@1"]["mean"]:.4f} MRR={r["mrr"]["mean"]:.4f}")

In [ ]:
print("
" + "=" * 70)
print("PHASE 2: λc Sweep (contrastive weight)")
print("=" * 70)
results["lc_sweep"] = {}
for lc in [0.0, 0.02, 0.05, 0.1, 0.15, 0.2, 0.3, 0.5]:
    print(f"  λc = {lc:.2f}...")
    results["lc_sweep"][f"lc_{lc:.2f}"] = run_exp(lc=lc, le=0.0, tag=f"lc={lc}")
    r = results["lc_sweep"][f"lc_{lc:.2f}"]
    print(f"    F1={r["id_f1"]["mean"]:.4f} Hard={r["hard_auc"]["mean"]:.4f} Hit@1={r["hit@1"]["mean"]:.4f}")

# Best λc by F1
best_lc = max(results["lc_sweep"].items(),
              key=lambda x: x[1]["id_f1"]["mean"])[1]["lc"]
print(f"
Best λc = {best_lc}")

In [ ]:
print("
" + "=" * 70)
print("PHASE 3: λe Sweep (entropy weight) with best λc")
print("=" * 70)
results["le_sweep"] = {}
for le in [0.0, 0.01, 0.02, 0.05, 0.1, 0.25]:
    print(f"  λe = {le:.2f} (λc={best_lc})...")
    results["le_sweep"][f"le_{le:.2f}"] = run_exp(lc=best_lc, le=le, tag=f"le={le}")
    r = results["le_sweep"][f"le_{le:.2f}"]
    print(f"    F1={r["id_f1"]["mean"]:.4f} Hard={r["hard_auc"]["mean"]:.4f} Hit@1={r["hit@1"]["mean"]:.4f}")

best_le = max(results["le_sweep"].items(),
              key=lambda x: x[1]["id_f1"]["mean"])[1]["le"]
print(f"
Best λe = {best_le}")

## Phase 4: Focal Loss + Extra Feat Ablation

In [ ]:
print("
" + "=" * 70)
print("PHASE 4: Focal Loss Sweep")
print("=" * 70)
results["focal_sweep"] = {}
for gamma in [1.0, 2.0, 3.0]:
    print(f"  Focal γ={gamma:.0f} (λc={best_lc}, λe={best_le})...")
    results["focal_sweep"][f"focal_{gamma:.0f}"] = run_exp(
        lc=best_lc, le=best_le, use_focal=True, gamma=gamma, tag=f"focal_g{gamma}"
    )
    r = results["focal_sweep"][f"focal_{gamma:.0f}"]
    print(f"    F1={r["id_f1"]["mean"]:.4f} Hard={r["hard_auc"]["mean"]:.4f}")
best_gamma = max(results["focal_sweep"].items(),
                 key=lambda x: x[1]["id_f1"]["mean"])[1]["gamma"]
use_focal = results["focal_sweep"][f"focal_{best_gamma:.0f}"]["id_f1"]["mean"] > results["baseline"]["id_f1"]["mean"]
print(f"
Best γ={best_gamma}, use_focal={use_focal}")

In [ ]:
print("
" + "=" * 70)
print("PHASE 5: Feature Ablation")
print("=" * 70)
results["feat_ablation"] = {}
for efd, name in [(0, "no_feat"), (1, "rep_only"), (2, "rep+freq")]:
    print(f"  {name} (λc={best_lc}, λe={best_le}, focal={use_focal})...")
    results["feat_ablation"][name] = run_exp(
        lc=best_lc, le=best_le, use_focal=use_focal, gamma=best_gamma,
        extra_feat_dim=efd, tag=name
    )
    r = results["feat_ablation"][name]
    print(f"    F1={r["id_f1"]["mean"]:.4f} Hit@1={r["hit@1"]["mean"]:.4f}")
best_efd = max(results["feat_ablation"].items(),
               key=lambda x: x[1]["id_f1"]["mean"])[1]["extra_feat_dim"]
print(f"
Best extra_feat_dim = {best_efd}")

## Phase 6: Final Model with Best Config

In [ ]:
print("
" + "=" * 70)
print("PHASE 6: Final UnifiedTMILv3 (Best Config)")
print("=" * 70)
results["final"] = run_exp(
    lc=best_lc, le=best_le, use_focal=use_focal, gamma=best_gamma,
    extra_feat_dim=best_efd, tag="final_v3_best"
)
r = results["final"]
print(f"
" + "=" * 70)
print("FINAL UNIFIED TMIL v3 RESULTS")
print("=" * 70)
print(f"ID-F1:    {r["id_f1"]["mean"]:.4f} ± {r["id_f1"]["std"]:.4f}  CI=[{r["id_f1"]["ci_low"]:.4f},{r["id_f1"]["ci_high"]:.4f}]")
print(f"ID-AUC:   {r["id_auc"]["mean"]:.4f} ± {r["id_auc"]["std"]:.4f}")
print(f"X-AUC:    {r["x_auc"]["mean"]:.4f} ± {r["x_auc"]["std"]:.4f}")
print(f"Hard-AUC: {r["hard_auc"]["mean"]:.4f} ± {r["hard_auc"]["std"]:.4f}")
print(f"Hit@1:    {r["hit@1"]["mean"]:.4f} ± {r["hit@1"]["std"]:.4f}")
print(f"Hit@5:    {r["hit@5"]["mean"]:.4f} ± {r["hit@5"]["std"]:.4f}")
print(f"Hit@10:   {r["hit@10"]["mean"]:.4f} ± {r["hit@10"]["std"]:.4f}")
print(f"MRR:      {r["mrr"]["mean"]:.4f} ± {r["mrr"]["std"]:.4f}")
print(f"Seeds:    {r["n_seeds"]}, Config: λc={best_lc}, λe={best_le}, focal={use_focal}(γ={best_gamma}), feat_dim={best_efd}")
print(f"Total:    {time.time()-t0:.0f}s")
print("=" * 70)

In [ ]:
# Save results
out = "/kaggle/working/results_v3.json"
os.makedirs(os.path.dirname(out), exist_ok=True)
save = {k: v for k, v in results.items()}
# Convert non-serializable
class NpEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, (np.floating,)): return float(obj)
        if isinstance(obj, (np.integer,)): return int(obj)
        if isinstance(obj, np.ndarray): return obj.tolist()
        return super().default(obj)
with open(out, "w") as f:
    json.dump(results, f, indent=2, cls=NpEncoder)
print(f"Results saved to {out}")